In [1]:
# CELL 1 — Load data
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/clean/groceries_clean.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df['first_order_date'] = pd.to_datetime(df['first_order_date'])

print('✅ Data loaded')
print(f'Rows: {df.shape[0]:,}')
print(f'Customers: {df["customer_id"].nunique():,}')
print(f'Date range: {df["order_date"].min().date()} → {df["order_date"].max().date()}')

✅ Data loaded
Rows: 38,765
Customers: 3,898
Date range: 2014-01-01 → 2015-12-30


In [2]:
# CELL 2 — Build RFM table
# Reference date = day after last transaction
reference_date = df['order_date'].max() + pd.Timedelta(days=1)

# Count orders per customer per date = 1 order per shopping trip
orders = df.groupby(['customer_id','order_date']).size().reset_index()
orders.columns = ['customer_id','order_date','items_bought']

rfm = orders.groupby('customer_id').agg(
    last_order_date = ('order_date', 'max'),
    frequency       = ('order_date', 'nunique'),
    monetary        = ('items_bought', 'sum')
).reset_index()

# Recency = days since last order
rfm['recency'] = (reference_date - rfm['last_order_date']).dt.days

print('✅ RFM table built')
print(f'Shape: {rfm.shape}')
print(f'\nRFM Summary:')
print(rfm[['recency','frequency','monetary']].describe().round(1))

✅ RFM table built
Shape: (3898, 5)

RFM Summary:
       recency  frequency  monetary
count   3898.0     3898.0    3898.0
mean     188.7        3.8       9.9
std      159.9        1.9       5.3
min        1.0        1.0       2.0
25%       58.0        2.0       6.0
50%      142.0        4.0       9.0
75%      281.0        5.0      13.0
max      728.0       11.0      36.0


In [3]:
# CELL 3 — Score each dimension 1-5 using quintiles
rfm['R_score'] = pd.qcut(rfm['recency'],   q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)

rfm['RFM_Score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print('✅ RFM scores computed')
print(f'\nScore distribution:')
print(rfm['RFM_Score'].value_counts().sort_index())

✅ RFM scores computed

Score distribution:
RFM_Score
3     316
4     214
5     281
6     284
7     299
8     343
9     346
10    348
11    346
12    312
13    315
14    288
15    206
Name: count, dtype: int64


In [4]:
# CELL 4 — Assign segments
def assign_segment(row):
    score = row['RFM_Score']
    r     = row['R_score']
    f     = row['F_score']
    if score >= 13:
        return 'Champion'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'Potential'
    elif r >= 3 and f <= 2:
        return 'At-Risk'
    else:
        return 'Churned'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

print('✅ Segments assigned')
print('\nSegment counts:')
seg_counts = rfm['segment'].value_counts()
print(seg_counts)
print(f'\nSegment %:')
print((seg_counts / len(rfm) * 100).round(1))

✅ Segments assigned

Segment counts:
segment
Loyal        1006
Potential     988
Churned       889
Champion      809
At-Risk       206
Name: count, dtype: int64

Segment %:
segment
Loyal        25.8
Potential    25.3
Churned      22.8
Champion     20.8
At-Risk       5.3
Name: count, dtype: float64


In [5]:
# CELL 5 — Segment summary stats
seg_summary = rfm.groupby('segment').agg(
    customers  = ('customer_id', 'count'),
    avg_recency   = ('recency',   'mean'),
    avg_frequency = ('frequency', 'mean'),
    avg_monetary  = ('monetary',  'mean'),
    avg_rfm_score = ('RFM_Score', 'mean')
).round(1).reset_index()

seg_summary['pct_customers'] = (seg_summary['customers'] / len(rfm) * 100).round(1)

print('📊 SEGMENT SUMMARY')
print('='*65)
print(seg_summary.to_string(index=False))

📊 SEGMENT SUMMARY
  segment  customers  avg_recency  avg_frequency  avg_monetary  avg_rfm_score  pct_customers
  At-Risk        206        131.6            1.7           4.1            5.5            5.3
 Champion        809         57.9            6.3          16.9           13.9           20.8
  Churned        889        399.9            2.0           4.7            4.2           22.8
    Loyal       1006        123.7            4.5          11.9           11.0           25.8
Potential        988        183.9            3.2           8.1            8.0           25.3


In [6]:
# CELL 6 — Visualization 1: Treemap of segments
fig1 = px.treemap(
    rfm,
    path=['segment'],
    values='RFM_Score',
    color='RFM_Score',
    color_continuous_scale='Reds',
    title='Customer Segments — Treemap<br><sup>Size = number of customers. Color = avg RFM score.</sup>'
)
fig1.update_layout(
    height=450,
    paper_bgcolor='#0F172A',
    font=dict(color='white', size=13),
    title_font=dict(size=16, color='white')
)
fig1.update_traces(textinfo='label+percent entry')
fig1.write_html('../outputs/rfm_treemap.html')
print('✅ Treemap saved!')

✅ Treemap saved!


In [7]:
# CELL 7 — Visualization 2: Scatter — Frequency vs Recency colored by segment
fig2 = px.scatter(
    rfm,
    x='recency',
    y='frequency',
    color='segment',
    size='monetary',
    hover_data=['customer_id','RFM_Score'],
    title='RFM Scatter — Recency vs Frequency<br><sup>Bubble size = total items bought. Color = segment.</sup>',
    color_discrete_map={
        'Champion':  '#DC2626',
        'Loyal':     '#F97316',
        'Potential': '#6C63DB',
        'At-Risk':   '#F59E0B',
        'Churned':   '#64748B'
    },
    labels={'recency':'Days Since Last Order','frequency':'Number of Orders'}
)
fig2.update_layout(
    height=500,
    plot_bgcolor='#0F172A',
    paper_bgcolor='#0F172A',
    font=dict(color='white', size=12),
    title_font=dict(size=16, color='white'),
    legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color='white'))
)
fig2.write_html('../outputs/rfm_scatter.html')
print('✅ Scatter saved!')

✅ Scatter saved!


In [8]:
# CELL 8 — Visualization 3: Bar chart — avg metrics per segment
fig3 = px.bar(
    seg_summary.sort_values('avg_rfm_score', ascending=True),
    x='avg_rfm_score',
    y='segment',
    orientation='h',
    color='avg_rfm_score',
    color_continuous_scale='Reds',
    text='customers',
    title='Segment Quality — Avg RFM Score<br><sup>Number on bar = customer count</sup>',
    labels={'avg_rfm_score':'Avg RFM Score','segment':'Segment'}
)
fig3.update_traces(texttemplate='%{text} customers', textposition='outside')
fig3.update_layout(
    height=400,
    plot_bgcolor='#0F172A',
    paper_bgcolor='#0F172A',
    font=dict(color='white', size=12),
    title_font=dict(size=16, color='white')
)
fig3.write_html('../outputs/rfm_segments_bar.html')
print('✅ Bar chart saved!')

✅ Bar chart saved!


In [9]:
# CELL 9 — Key Insights
print('='*60)
print('👥 RFM SEGMENTATION — KEY INSIGHTS')
print('='*60)

for _, row in seg_summary.sort_values('avg_rfm_score', ascending=False).iterrows():
    print(f'\n🎯 {row["segment"]} ({row["pct_customers"]}% of customers)')
    print(f'   Count:         {row["customers"]:,} customers')
    print(f'   Avg Recency:   {row["avg_recency"]:.0f} days since last order')
    print(f'   Avg Frequency: {row["avg_frequency"]:.1f} orders')
    print(f'   Avg Items:     {row["avg_monetary"]:.1f} items bought total')

👥 RFM SEGMENTATION — KEY INSIGHTS

🎯 Champion (20.8% of customers)
   Count:         809 customers
   Avg Recency:   58 days since last order
   Avg Frequency: 6.3 orders
   Avg Items:     16.9 items bought total

🎯 Loyal (25.8% of customers)
   Count:         1,006 customers
   Avg Recency:   124 days since last order
   Avg Frequency: 4.5 orders
   Avg Items:     11.9 items bought total

🎯 Potential (25.3% of customers)
   Count:         988 customers
   Avg Recency:   184 days since last order
   Avg Frequency: 3.2 orders
   Avg Items:     8.1 items bought total

🎯 At-Risk (5.3% of customers)
   Count:         206 customers
   Avg Recency:   132 days since last order
   Avg Frequency: 1.7 orders
   Avg Items:     4.1 items bought total

🎯 Churned (22.8% of customers)
   Count:         889 customers
   Avg Recency:   400 days since last order
   Avg Frequency: 2.0 orders
   Avg Items:     4.7 items bought total


In [10]:
# CELL 10 — Save + complete
rfm.to_csv('../data/clean/rfm_segments.csv', index=False)
seg_summary.to_csv('../data/clean/rfm_summary.csv', index=False)

print('🎉 DAY 5 COMPLETE!')
print('='*55)
print('What you built:')
print('  ✅ RFM scores for 3,898 customers')
print('  ✅ 5 segments: Champion, Loyal, Potential, At-Risk, Churned')
print('  ✅ Treemap → rfm_treemap.html')
print('  ✅ Scatter → rfm_scatter.html')
print('  ✅ Bar chart → rfm_segments_bar.html')
print('  ✅ Segment stats saved')
print('='*55)
print('Day 6 tomorrow: Cohort Retention Analysis 📈')

🎉 DAY 5 COMPLETE!
What you built:
  ✅ RFM scores for 3,898 customers
  ✅ 5 segments: Champion, Loyal, Potential, At-Risk, Churned
  ✅ Treemap → rfm_treemap.html
  ✅ Scatter → rfm_scatter.html
  ✅ Bar chart → rfm_segments_bar.html
  ✅ Segment stats saved
Day 6 tomorrow: Cohort Retention Analysis 📈
